In [1]:
from pathlib import Path

PROJECT_DIR = Path("~/UltraCNN").expanduser()
RUNS_DIR = PROJECT_DIR / "runs" / "detect"

runs = sorted(
    RUNS_DIR.glob("kitti_yolo11n*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

for run in runs:
    print(f"\nRun: {run.name}")
    print("Best:", (run / "weights" / "best.pt").exists())
    print("Last:", (run / "weights" / "last.pt").exists())
    print("Results:", (run / "results.csv").exists())


Run: kitti_yolo11n
Best: True
Last: True
Results: True


In [2]:
from ultralytics import YOLO

TRAIN_RUN = runs[0]

BEST_MODEL_PATH = TRAIN_RUN / "weights" / "best.pt"

assert BEST_MODEL_PATH.exists(), "best.pt not found!"

finetuned_model = YOLO(str(BEST_MODEL_PATH))

print("Loaded:", BEST_MODEL_PATH)

Loaded: /home/ghost/UltraCNN/runs/detect/kitti_yolo11n/weights/best.pt


In [3]:
from pathlib import Path

PROJECT_DIR = Path("~/UltraCNN").expanduser()

DATASET = PROJECT_DIR / "data/kitti/tracking"

OUTPUT_DIR = DATASET / "yolo_dataset_v1"

yaml_path = OUTPUT_DIR / "data.yaml"

assert yaml_path.exists(), f"Dataset YAML not found: {yaml_path}"

print("Dataset configuration:", yaml_path)
print(yaml_path.read_text())

Dataset configuration: /home/ghost/UltraCNN/data/kitti/tracking/yolo_dataset_v1/data.yaml
path: /home/ghost/UltraCNN/data/kitti/tracking/yolo_dataset_v1
train: images/train
val: images/val
names:
  0: car
  1: truck
  2: pedestrian
  3: cyclist



In [4]:
metrics = finetuned_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=4,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name="kitti_finetuned_validation"
)

print("\nFINE-TUNED YOLO11n RESULTS")
print("-" * 40)

print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

Ultralytics 8.4.184 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1650, 3716MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 414.7±241.0 MB/s, size: 758.8 KB)
val: Scanning /home/ghost/UltraCNN/data/kitti/tracking/yolo_dataset_v1/labels/val.cache... 2380 images, 38 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2380/2380 277.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 595/595 19.2it/s 31.0s0.1ss
                   all       2380      16509      0.536      0.411      0.445       0.25
                   car       2026       9085      0.875      0.656      0.772      0.516
                 truck        145        145     0.0755     0.0828     0.0129    0.00941
            pedestrian       1196       6870      0.836      0.425       0.58       0.24
               cyclist        287        409      0.358      0.479   

In [4]:
from pathlib import Path

PROJECT_DIR = Path("~/UltraCNN").expanduser()

DATASET = PROJECT_DIR / "data/kitti/tracking"

OUTPUT_DIR = DATASET / "yolo_dataset_v1"

yaml_path = OUTPUT_DIR / "data.yaml"

CLASS_NAMES = [
    "car",
    "truck",
    "pedestrian",
    "cyclist"
]

CLASS_MAP = {
    "Car": 0,
    "Van": 0,
    "Truck": 1,
    "Pedestrian": 2,
    "Cyclist": 3
}

assert yaml_path.exists(), "data.yaml not found!"

print("Classes:", CLASS_NAMES)
print("Dataset YAML:", yaml_path)

Classes: ['car', 'truck', 'pedestrian', 'cyclist']
Dataset YAML: /home/ghost/UltraCNN/data/kitti/tracking/yolo_dataset_v1/data.yaml


In [7]:
print("PER-CLASS RESULTS")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:12s} | "
        f"mAP50: {metrics.box.ap50[i]:.4f} | "
        f"mAP50-95: {metrics.box.ap[i]:.4f}"
    )

PER-CLASS RESULTS
car          | mAP50: 0.7716 | mAP50-95: 0.5157
truck        | mAP50: 0.0129 | mAP50-95: 0.0094
pedestrian   | mAP50: 0.5801 | mAP50-95: 0.2400
cyclist      | mAP50: 0.4148 | mAP50-95: 0.2353
